# ## test pipeline implementation


In [4]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
SRC_PATH = PROJECT_ROOT / "src"

sys.path.insert(0, str(SRC_PATH))

print("Project root:", PROJECT_ROOT)
print("SRC path:", SRC_PATH)
print("SRC exists:", SRC_PATH.exists())

Project root: c:\Users\Sagar\Documents\expernetic-airbnb-data-engineering\expernetic-airbnb-data-engineering
SRC path: c:\Users\Sagar\Documents\expernetic-airbnb-data-engineering\expernetic-airbnb-data-engineering\src
SRC exists: True


In [5]:
from ingestion.load_data import load_city_data

In [6]:
listings, reviews, calendar = load_city_data("london")

print("Listings:", listings.shape)
print("Reviews:", reviews.shape)
print("Calendar:", calendar.shape)

Listings: (96871, 79)
Reviews: (2097996, 6)
Calendar: (35357974, 7)


In [7]:
from profiling.validate_data import validate_data

validate_data(
    listings,
    reviews,
    calendar
)

Running validation...
Validation passed.


In [8]:
from cleaning.clean_listings import clean_listings

cleaned_listings = clean_listings(listings)

print(cleaned_listings[[
    "price",
    "price_clean",
    "price_outlier",
    "price_outlier_threshold",
    "host_tenure_years"
]].head())

     price  price_clean  price_outlier  price_outlier_threshold  \
0   $70.00         70.0          False                   1100.0   
1  $149.00        149.0          False                   1100.0   
2  $411.00        411.0          False                   1100.0   
3      NaN          NaN          False                   1100.0   
4  $210.00        210.0          False                   1100.0   

   host_tenure_years  
0          15.843836  
1          15.791781  
2          15.709589  
3          15.983562  
4          15.315068  


In [9]:
from transformation.review_aggregation import aggregate_reviews

review_summary = aggregate_reviews(reviews)

print(review_summary.shape)
print(review_summary.head())

(72749, 4)
   listing_id  total_reviews first_review_date last_review_date
0       13913             55        2010-08-18       2025-08-21
1       15400             97        2009-12-21       2025-04-05
2       17402             56        2011-03-21       2024-02-19
3       24328             95        2010-11-15       2025-07-05
4       36274             15        2011-03-20       2025-09-06


In [10]:
from transformation.calendar_aggregation import aggregate_calendar

calendar_summary = aggregate_calendar(calendar)

print(calendar_summary.shape)
print(calendar_summary.head())

(96871, 6)
   listing_id  total_days  available_days  unavailable_days  \
0       13913         365             331                34   
1       15400         365             199               166   
2       17402         365              80               285   
3       24328         365             294                71   
4       36274         365             323                42   

   availability_rate  occupancy_rate  
0           0.906849        0.093151  
1           0.545205        0.454795  
2           0.219178        0.780822  
3           0.805479        0.194521  
4           0.884932        0.115068  


In [11]:
from transformation.build_master import build_master_dataset

city_master = build_master_dataset(
    cleaned_listings,
    review_summary,
    calendar_summary,
    "London"
)

print(city_master.shape)
print(city_master[[
    "city",
    "id",
    "price_clean",
    "total_reviews",
    "availability_rate",
    "occupancy_rate"
]].head())

(96871, 92)
     city     id  price_clean  total_reviews  availability_rate  \
0  London  13913         70.0           55.0           0.906849   
1  London  15400        149.0           97.0           0.545205   
2  London  17402        411.0           56.0           0.219178   
3  London  24328          NaN           95.0           0.805479   
4  London  36274        210.0           15.0           0.884932   

   occupancy_rate  
0        0.093151  
1        0.454795  
2        0.780822  
3        0.194521  
4        0.115068  


In [12]:
from pipeline import run_city_pipeline

london_pipeline_master = run_city_pipeline(
    city_folder="london",
    city_name="London"
)

print(london_pipeline_master.shape)


Running validation...
Validation passed.
Saved: data\processed\london_master.csv
(96871, 92)


In [13]:
print(london_pipeline_master.head())

      id                         listing_url       scrape_id last_scraped  \
0  13913  https://www.airbnb.com/rooms/13913  20250914034649   2025-09-16   
1  15400  https://www.airbnb.com/rooms/15400  20250914034649   2025-09-16   
2  17402  https://www.airbnb.com/rooms/17402  20250914034649   2025-09-16   
3  24328  https://www.airbnb.com/rooms/24328  20250914034649   2025-09-18   
4  36274  https://www.airbnb.com/rooms/36274  20250914034649   2025-09-15   

            source                                               name  \
0      city scrape                Holiday London DB Room Let-on going   
1      city scrape                Bright Chelsea  Apartment. Chelsea!   
2      city scrape   Very Central Modern 3-Bed/2 Bath By Oxford St W1   
3  previous scrape                   Battersea live/work artist house   
4      city scrape  Bright 1 bedroom apt off brick lane in Shoreditch   

                                         description  \
0  My bright double bedroom with a large w

In [14]:
from pipeline import run_city_pipeline

ny_pipeline_master = run_city_pipeline(
    city_folder="new_york",
    city_name="New York"
)

print(ny_pipeline_master.shape)


Running validation...
Validation passed.
Saved: data\processed\new_york_master.csv
(35036, 103)


In [15]:
from pipeline import run_city_pipeline

london_pipeline_master = run_city_pipeline("london", "London")
print(london_pipeline_master.shape)



Running validation...
Validation passed.
Saved: data\processed\london_master.csv
(96871, 92)


In [16]:
from pipeline import run_city_pipeline
import pandas as pd

london_pipeline_master = run_city_pipeline("london", "London")
ny_pipeline_master = run_city_pipeline("new_york", "New York")

combined_pipeline_master = pd.concat(
    [london_pipeline_master, ny_pipeline_master],
    ignore_index=True
)

combined_pipeline_master.to_csv(
    "../data/processed/pipeline_master_dataset.csv",
    index=False
)

print(combined_pipeline_master.shape)
print(combined_pipeline_master["city"].value_counts())

Running validation...
Validation passed.
Saved: data\processed\london_master.csv
Running validation...
Validation passed.
Saved: data\processed\new_york_master.csv
(131907, 103)
city
London      96871
New York    35036
Name: count, dtype: int64
